## В этом домашнем задании вы сделаете первые шаги в мире линейной бинарной классификации!

In [1]:
import pandas as pd
import numpy as np
import warnings

#### Задание 1

Мы будем работать с данными **Microsoft Malware Detection**

Таргетом будет последний столбец `HasDetection`, который принимает значения $\{0,\, 1\}$ в случае отсутствия или наличия вируса на компьютере соответственно. Признаками будут выступать всевозможные характеристики компьютера.

In [3]:
data = pd.read_csv('train.csv', low_memory=False)

In [4]:
data.head()

,MachineIdentifier,ProductName,EngineVersion,AppVersion,AvSigVersion,IsBeta,RtpStateBitfield,IsSxsPassiveMode,DefaultBrowsersIdentifier,AVProductStatesIdentifier,...,Census_FirmwareVersionIdentifier,Census_IsSecureBootEnabled,Census_IsWIMBootEnabled,Census_IsVirtualDevice,Census_IsTouchEnabled,Census_IsPenCapable,Census_IsAlwaysOnAlwaysConnectedCapable,Wdft_IsGamer,Wdft_RegionIdentifier,HasDetections
0,0000028988387b115f69f31a3bf04f09,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1735.0,0,7.0,0,NaN,53447.0,...,36144.0,0,NaN,0.0,0,0,0.0,0.0,10.0,0
1,000007535c3f730efa9ea0b7ef1bd645,win8defender,1.1.14600.4,4.13.17134.1,1.263.48.0,0,7.0,0,NaN,53447.0,...,57858.0,0,NaN,0.0,0,0,0.0,0.0,8.0,0
2,000007905a28d863f6d0d597892cd692,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1341.0,0,7.0,0,NaN,53447.0,...,52682.0,0,NaN,0.0,0,0,0.0,0.0,3.0,0
3,00000b11598a75ea8ba1beea8459149f,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1527.0,0,7.0,0,NaN,53447.0,...,20050.0,0,NaN,0.0,0,0,0.0,0.0,3.0,1
4,000014a5f00daa18e76b81417eeb99fc,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1379.0,0,7.0,0,NaN,53447.0,...,19844.0,0,0.0,0.0,0,0,0.0,0.0,1.0,1


In [5]:
data[['ProductName', 'MachineIdentifier']]

,ProductName,MachineIdentifier
0,win8defender,0000028988387b115f69f31a3bf04f09
1,win8defender,000007535c3f730efa9ea0b7ef1bd645
2,win8defender,000007905a28d863f6d0d597892cd692
3,win8defender,00000b11598a75ea8ba1beea8459149f
4,win8defender,000014a5f00daa18e76b81417eeb99fc
...,...,...
199995,win8defender,05c15aa6487f9710169e37c749a10441
199996,win8defender,05c15b7b1b0f9f19a3279eee6572b3cd
199997,win8defender,05c15e17f59c0e298c06dea9ed3a3ab4
199998,win8defender,05c15e1affb2558420062f2d994f90d1


In [6]:
data.columns

Index(['MachineIdentifier', 'ProductName', 'EngineVersion', 'AppVersion',
       'AvSigVersion', 'IsBeta', 'RtpStateBitfield', 'IsSxsPassiveMode',
       'DefaultBrowsersIdentifier', 'AVProductStatesIdentifier',
       'AVProductsInstalled', 'AVProductsEnabled', 'HasTpm',
       'CountryIdentifier', 'CityIdentifier', 'OrganizationIdentifier',
       'GeoNameIdentifier', 'LocaleEnglishNameIdentifier', 'Platform',
       'Processor', 'OsVer', 'OsBuild', 'OsSuite', 'OsPlatformSubRelease',
       'OsBuildLab', 'SkuEdition', 'IsProtected', 'AutoSampleOptIn', 'PuaMode',
       'SMode', 'IeVerIdentifier', 'SmartScreen', 'Firewall', 'UacLuaenable',
       'Census_MDC2FormFactor', 'Census_DeviceFamily',
       'Census_OEMNameIdentifier', 'Census_OEMModelIdentifier',
       'Census_ProcessorCoreCount', 'Census_ProcessorManufacturerIdentifier',
       'Census_ProcessorModelIdentifier', 'Census_ProcessorClass',
       'Census_PrimaryDiskTotalCapacity', 'Census_PrimaryDiskTypeName',
       'Census_

Удалите константные признаки и признаки `ProductName` `MachineIdentifier`

In [7]:
constant_features = []
numeric_columns = data.loc[:,data.dtypes!=object].columns
for col in numeric_columns:
    if np.max(data[col]) == np.min(data[col]):
        constant_features.append(col)

print(constant_features)

['IsBeta', 'Census_IsWIMBootEnabled']


In [8]:
### Your code is here
data = data.drop(['ProductName', 'MachineIdentifier','IsBeta', 'Census_IsWIMBootEnabled'], axis=1)






Посмотрите на соотношение классов в таргете. Все ли хорошо?

In [9]:
print(sum(data['HasDetections'] == 1), '- positive class,')
print(sum(data['HasDetections'] == 0), '- negative class')

100060 - positive class,
99940 - negative class


Ответьте на вопрос: почему с вашей точки зрения важно иметь представление о балансе классов в ваших данных?

Избавьтесь от пропусков в данных! 

Новый для нас прием: признаки с более чем половиной пропусков следует удалить.

Согласитесь, если в вашей колонке среди 100 объектов всего лишь у 2 есть какое-то непропущенное значение, странно все остальные заполнять средним от этих двух чисел. Такие "редкие" признаки лучше вообще опустить!


В категориальных колонках заменим отсутствующую категорию просто некоторой новой и назовем ее `NaN`.

А в числовых, ради разнообразия, заполним пропуски медианным значением.

In [10]:
half_lenght = len(data) / 2
half_lenght

100000.0

In [11]:
more_half=[]
for col in data.columns:
    if data[col].isna().sum() >= half_lenght:
        more_half.append(col)
more_half       

['DefaultBrowsersIdentifier',
 'PuaMode',
 'Census_ProcessorClass',
 'Census_InternalBatteryType',
 'Census_IsFlightingInternal',
 'Census_ThresholdOptIn']

In [12]:
data = data.drop(more_half, axis=1)
data.shape

(200000, 73)

In [13]:
### Your code is here
list_nan=[]
for col in data.columns:
    if data[col].isna().sum()>0:
        list_nan.append(col)
        print(f" в колонке {col} {data[col].isna().sum()} пропусков , тип {data[col].dtypes}")

 в колонке RtpStateBitfield 764 пропусков , тип float64
 в колонке AVProductStatesIdentifier 804 пропусков , тип float64
 в колонке AVProductsInstalled 804 пропусков , тип float64
 в колонке AVProductsEnabled 804 пропусков , тип float64
 в колонке CityIdentifier 7215 пропусков , тип float64
 в колонке OrganizationIdentifier 61537 пропусков , тип float64
 в колонке GeoNameIdentifier 1 пропусков , тип float64
 в колонке OsBuildLab 1 пропусков , тип object
 в колонке IsProtected 799 пропусков , тип float64
 в колонке SMode 11863 пропусков , тип float64
 в колонке IeVerIdentifier 1362 пропусков , тип float64
 в колонке SmartScreen 71123 пропусков , тип object
 в колонке Firewall 2078 пропусков , тип float64
 в колонке UacLuaenable 248 пропусков , тип float64
 в колонке Census_OEMNameIdentifier 2082 пропусков , тип float64
 в колонке Census_OEMModelIdentifier 2245 пропусков , тип float64
 в колонке Census_ProcessorCoreCount 922 пропусков , тип float64
 в колонке Census_ProcessorManufacturer

In [14]:
numeric_columns = data.loc[:,data.dtypes!=object].columns
for col in numeric_columns:
    data[col]=data[col].fillna(data[col].median())
print(data[numeric_columns].isna().sum())

RtpStateBitfield                                     0
IsSxsPassiveMode                                     0
AVProductStatesIdentifier                            0
AVProductsInstalled                                  0
AVProductsEnabled                                    0
HasTpm                                               0
CountryIdentifier                                    0
CityIdentifier                                       0
OrganizationIdentifier                               0
GeoNameIdentifier                                    0
LocaleEnglishNameIdentifier                          0
OsBuild                                              0
OsSuite                                              0
IsProtected                                          0
AutoSampleOptIn                                      0
SMode                                                0
IeVerIdentifier                                      0
Firewall                                             0
UacLuaenab

In [15]:
### 1 ВАРИАНТ КАК ОПРЕДЕЛИТЬ КАТЕГОРИАЛЬНЫЕ КОЛОНКИ

categorical_columns = data.select_dtypes(include=['object', 'category']).columns
data[categorical_columns]
for col in categorical_columns:
    data[col]=data[col].fillna('NaN')
print(data[categorical_columns].isna().sum())

EngineVersion                       0
AppVersion                          0
AvSigVersion                        0
Platform                            0
Processor                           0
OsVer                               0
OsPlatformSubRelease                0
OsBuildLab                          0
SkuEdition                          0
SmartScreen                         0
Census_MDC2FormFactor               0
Census_DeviceFamily                 0
Census_PrimaryDiskTypeName          0
Census_ChassisTypeName              0
Census_PowerPlatformRoleName        0
Census_OSVersion                    0
Census_OSArchitecture               0
Census_OSBranch                     0
Census_OSEdition                    0
Census_OSSkuName                    0
Census_OSInstallTypeName            0
Census_OSWUAutoUpdateOptionsName    0
Census_GenuineStateName             0
Census_ActivationChannel            0
Census_FlightRing                   0
dtype: int64


In [16]:
### 2 ВАРИАНТ КАК ОПРЕДЕЛИТЬ КАТЕГОРИАЛЬНЫЕ КОЛОНКИ
categorical_columns = data.loc[:,data.dtypes==object].columns
for col in categorical_columns:
    data[col]=data[col].fillna('NaN')
print(data[categorical_columns].isna().sum())

EngineVersion                       0
AppVersion                          0
AvSigVersion                        0
Platform                            0
Processor                           0
OsVer                               0
OsPlatformSubRelease                0
OsBuildLab                          0
SkuEdition                          0
SmartScreen                         0
Census_MDC2FormFactor               0
Census_DeviceFamily                 0
Census_PrimaryDiskTypeName          0
Census_ChassisTypeName              0
Census_PowerPlatformRoleName        0
Census_OSVersion                    0
Census_OSArchitecture               0
Census_OSBranch                     0
Census_OSEdition                    0
Census_OSSkuName                    0
Census_OSInstallTypeName            0
Census_OSWUAutoUpdateOptionsName    0
Census_GenuineStateName             0
Census_ActivationChannel            0
Census_FlightRing                   0
dtype: int64


Создайте копию полученного датафрейма и положите ее в переменную data_2. Понадобится в следующих заданиях.

In [17]:
data.isna().sum().sum()

0

In [18]:
data_2 = data.copy()
data_2.shape[1]

73

Так же поработаем над всеми категориальными колонкам перед запуском непосредственно моделей.

Провернем самый базовый и наглый метод - несмотря на количество уникальных значений в каждой категории, просто применим ко всей категориальной части датасета `OneHotEncoding`

In [19]:
### Your code is here
for col in categorical_columns :
    one_hot=pd.get_dummies(data[col], prefix=col, drop_first=True)
    data = pd.concat((data.drop(col, axis=1), one_hot), axis=1)
print(len(data.columns))
print(data.shape[1])


6114
6114


###  Разделим выборку на тренировочную и тестовую

P.S. в задачах классификации, как и в задаче регрессии, можно использовать технологию Кросс-Валидации. Например, по одному из двух следующих сценариев:

1) Отделить валидацию и тест, произвести подбор лучшей модели с помощью `K-Fold` на валидации, финально обучить выбранную модель на всей валидации и замерить качество на заранее отложенном финальном тесте!

2) Всю выборку назвать валидационной и на ней применить `K-Fold` без финального замера.

В этой домашней работе попросим Вас быть еще проще! :)
Реализуем просто технологию отложенной выборки в пропорции 3:1

In [20]:
from sklearn.model_selection import train_test_split 

X = data.drop(columns=['HasDetections'])
y = data['HasDetections']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.25,
                                                    shuffle=True,
                                                    random_state=1)

Соберите `Pipeline`, реализовав в нем 2 шага: стандартизация данных через `StandardScaler` и обучение логистической регрессии с помощью `LogisticRegression`, положите результаты в переменную `pipe`, а в классе модели `LogisticRegression` укажите параметр `penalty='none'`

In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

### Your code is here

pipe = Pipeline([('scaler', StandardScaler()),
                 ('LR', LogisticRegression(penalty=None))])

pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)


0.62246

Чтобы замерить качество работы такой модели на трейне и на тесте воспользуемся функцией `cross_validate`

Вопрос: что передавать ей в параметр cv? Ведь мы уже разделили нашу выборку на трейн и тест (имеем всего 1 fold). Для этого можно просто передать список от кортежа, содержащего индексы тренировочных и тестовых объектов.

In [22]:
from sklearn.model_selection import cross_validate
import datetime

custom_cv = [(X_train.index.to_list(), X_test.index.to_list())]

begin_time = datetime.datetime.now()

cv_result_pipe = cross_validate(pipe, X, y, scoring='accuracy',
                                cv=custom_cv, return_train_score=True)


print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']).round(3)}")

print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")

Accuracy на трейне: 0.651
Accuracy на тесте: 0.622
Время работы алгоритма: 0:02:27.082216


Что можете сказать про время работы алгоритма?

Очевидно, оно достаточно большие. Уж тем более для линейных моделей.

Такое как раз-таки происходит из-за того, что количество фичей, который мы передали нашей модели - гигантское! Классу требуется много времени и памяти, чтобы обработать датасет.

Поэтому те колонки, в которых количество уникальных категорий превышает какое-то адекватное число, следует кодировать иначе, нежели с помощью технологии `One-Hot-Encoding`.

Теперь вы верите, что более умные кодировки зачастую прям необходимы! Раньше мы этот факт не демонстрировали!

Дело еще вот в чем: в классе `LogisticRegression`, как и, например, `Lasso`, есть параметр, ограничивающий максимальное количество итераций во время обучения модели. Так, если данных много и итераций тоже ожидается большое число, найденная разделяющая гиперплоскость может оказаться не самой лучшей, так как наш алгоритм (будь то градиентный спуск или любой иной) просто 'не доползет'. 

#### Задание 2

Теперь попробуем другой метод кодирования категориальных колонок, а именно счётчики.
Построем ту же модель и на том же разделении, просто заново иначе переобработаем датасет. 

Для тех категориальных признаков, у которых количество уникальных значений в колоночках больше 5, применим `MeanTargetEncoding`.

Для остальных оставим любимый `OneHotEncoding` (как делали на практике и в предыдущем уроке).

In [23]:
### Your code is here
data_2.shape[1]
for col in categorical_columns :
    if data_2[col].nunique() <= 5:
        one_hot=pd.get_dummies(data_2[col], prefix=col, drop_first=True)
        data_2 = pd.concat((data_2.drop(col, axis=1), one_hot), axis=1)
    else:
        mean_target = data_2.groupby(col)['HasDetections'].mean()
        data_2[col] = data_2[col].map(mean_target)
        
print(len(data_2.columns))
print(data_2.shape[1])



82
82


In [24]:
X_2 = data_2.drop(columns=['HasDetections'])
y_2 = data_2['HasDetections']

Опять обучим модель, пока что без изменений! Скажите, стало ли быстрее? А что с качеством?

In [25]:
### Your code is here


custom_cv = [(X_train.index.to_list(), X_test.index.to_list())]

begin_time = datetime.datetime.now()

cv_result_pipe = cross_validate(pipe, X_2, y_2, scoring='accuracy',
                                cv=custom_cv, return_train_score=True)


print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']).round(3)}")

print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")





Accuracy на трейне: 0.638
Accuracy на тесте: 0.638
Время работы алгоритма: 0:00:01.778222


#### Задание 3: Регуляризация

Как и в моделях регрессии, решая задачу классификации, можем штрафовать минимизируемый функционал за большие веса, добавив к нему L1 или L2 норму весов (все как раньше!).

Для этого в изначальном классе `LogisticRegression` изменить параметр `penalty` на l1 или l2 соответственно. Выберите второй вариант! Можно воспользоваться методом `set_params` и применить его к `pipe`.

In [26]:
### Your code is here
X_train, X_test, y_train, y_test = train_test_split(X_2, y_2,
                                                    test_size=0.25,
                                                    shuffle=True,
                                                    random_state=1)
custom_cv = [(X_train.index.to_list(), X_test.index.to_list())]

pipe = Pipeline([('scaler', StandardScaler()),
                 ('LR', LogisticRegression(penalty='l2'))])

pipe.fit(X_train, y_train)
pipe.score(X_test, y_test)


begin_time = datetime.datetime.now()

cv_result_pipe = cross_validate(pipe, X_2, y_2, scoring='accuracy',
                                cv=custom_cv, return_train_score=True)


print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']).round(3)}")

print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")

Accuracy на трейне: 0.638
Accuracy на тесте: 0.638
Время работы алгоритма: 0:00:01.654706


Теперь наша модель будет строить логистическую регрессию с L2 регуляризатором! Помним, что у регуляризируемых моделей есть гиперпараметр, контролирующий силу регуляризации, который выбирается ДО запуска метода fit, то есть заранее, когда модель еще не обучена. 

Конечно же, выбор этого параметра влияет итоговые результаты. Хочется поперебирать различные параметры регуляризации и найти такой, при котором качество на тесте окажется лучшим! 

Сгенерируем массив гиперпараметра, которые планируем перебрать:

In [27]:
alphas = np.linspace(0.0001, 0.01, 100)
alphas

array([0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008,
       0.0009, 0.001 , 0.0011, 0.0012, 0.0013, 0.0014, 0.0015, 0.0016,
       0.0017, 0.0018, 0.0019, 0.002 , 0.0021, 0.0022, 0.0023, 0.0024,
       0.0025, 0.0026, 0.0027, 0.0028, 0.0029, 0.003 , 0.0031, 0.0032,
       0.0033, 0.0034, 0.0035, 0.0036, 0.0037, 0.0038, 0.0039, 0.004 ,
       0.0041, 0.0042, 0.0043, 0.0044, 0.0045, 0.0046, 0.0047, 0.0048,
       0.0049, 0.005 , 0.0051, 0.0052, 0.0053, 0.0054, 0.0055, 0.0056,
       0.0057, 0.0058, 0.0059, 0.006 , 0.0061, 0.0062, 0.0063, 0.0064,
       0.0065, 0.0066, 0.0067, 0.0068, 0.0069, 0.007 , 0.0071, 0.0072,
       0.0073, 0.0074, 0.0075, 0.0076, 0.0077, 0.0078, 0.0079, 0.008 ,
       0.0081, 0.0082, 0.0083, 0.0084, 0.0085, 0.0086, 0.0087, 0.0088,
       0.0089, 0.009 , 0.0091, 0.0092, 0.0093, 0.0094, 0.0095, 0.0096,
       0.0097, 0.0098, 0.0099, 0.01  ])

Чтобы отобрать среди данного массива гиперпараметров лучший, воспользуйтесь конструкцией `GridSearchCV` из `sklearn`

In [28]:
from sklearn.model_selection import GridSearchCV

param_grid ={
    'LR__C': alphas
}

### Your code is here
search = GridSearchCV(pipe, param_grid, cv=custom_cv, scoring='accuracy')
search.fit(X_2, y_2)


print(f"Best parameter (CV score={search.best_score_:.5f}):")
print(search.best_params_)

Best parameter (CV score=0.63842):
{'LR__C': 0.0006000000000000001}


In [29]:
search.cv_results_

{'mean_fit_time': array([0.81945157, 0.86249566, 0.8845315 , 0.94294834, 0.93801951,
        0.9385426 , 0.93298244, 0.94467187, 0.94829798, 0.98749781,
        0.92367578, 0.97030234, 0.99121928, 0.98425937, 1.05007744,
        1.01558971, 1.04132915, 1.03353953, 1.00476813, 1.0113287 ,
        1.02012277, 1.0264647 , 0.99971104, 1.01366901, 1.08970976,
        1.18778706, 1.04495955, 1.06731033, 1.06578612, 1.01611757,
        1.02012372, 0.99114347, 0.98929214, 0.9870863 , 1.00332379,
        1.01522541, 1.00015807, 1.00972056, 1.00027823, 1.02538776,
        1.00050592, 1.01620388, 1.01329136, 0.98466825, 1.01485133,
        1.00110269, 0.98452902, 1.01085162, 0.98711514, 1.00279784,
        1.07534266, 1.13107085, 0.98239779, 1.16900468, 0.99826336,
        1.12527871, 1.05759358, 1.09280109, 1.00361872, 0.99951315,
        1.03453207, 0.99953508, 1.07065058, 1.05527139, 0.99515462,
        1.04008698, 1.06530499, 1.04875803, 1.01083636, 0.97494578,
        0.98478985, 1.00564265,